# Register Tuned RandomForest Model in MLflow
This notebook finds the latest tuned RandomForest run from the experiment and registers its model artifact in MLflow Model Registry.

In [1]:
from pathlib import Path
import time
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "iomt_ml.heartdisease.basemodels_tuned.tuned"
MODEL_FAMILY_TAG = "RandomForest (tuned)"
REGISTERED_MODEL_NAME = "HeartDiseaseRandomForestTuned"
MODEL_ALIAS = "latest"

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

print("Tracking URI:", TRACKING_URI)
print("Experiment:", EXPERIMENT_NAME)
print("Target registry name:", REGISTERED_MODEL_NAME)

Tracking URI: http://127.0.0.1:5000
Experiment: iomt_ml.heartdisease.basemodels_tuned.tuned
Target registry name: HeartDiseaseRandomForestTuned


In [2]:
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    raise RuntimeError(f"Experiment not found: {EXPERIMENT_NAME}")

runs = client.search_runs(
    [exp.experiment_id],
    filter_string=f"tags.model_family = '{MODEL_FAMILY_TAG}'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)

if not runs:
    raise RuntimeError("No tuned RandomForest runs were found. Run the basemodel logging cell first.")

run = runs[0]
run_id = run.info.run_id
model_artifact_name = run.data.tags.get("model_artifact_name", "model")
model_uri = f"runs:/{run_id}/{model_artifact_name}"

print("Selected run_id:", run_id)
print("Artifact path:", model_artifact_name)
print("Model URI to register:", model_uri)

Selected run_id: 422307d7b79d47648248e36a99438a17
Artifact path: randomforest_tuned_20260330T220816Z_a71ed44a
Model URI to register: runs:/422307d7b79d47648248e36a99438a17/randomforest_tuned_20260330T220816Z_a71ed44a


In [3]:
mv = mlflow.register_model(model_uri=model_uri, name=REGISTERED_MODEL_NAME)
print("Registration requested. Version:", mv.version)

# Optional wait loop for local file-store backends
for _ in range(30):
    cur = client.get_model_version(name=REGISTERED_MODEL_NAME, version=mv.version)
    if cur.status == "READY":
        break
    time.sleep(1)

final_mv = client.get_model_version(name=REGISTERED_MODEL_NAME, version=mv.version)
print("Final status:", final_mv.status)

# Keep an alias pointing to the newest version (works in MLflow 2.9+)
try:
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias=MODEL_ALIAS,
        version=final_mv.version,
    )
    print(f"Alias '{MODEL_ALIAS}' -> version {final_mv.version}")
except Exception as e:
    print("Alias not set (this may be expected on older MLflow versions):", e)

print("Registered model name:", REGISTERED_MODEL_NAME)
print("Registered model version:", final_mv.version)
print("Source run id:", run_id)

Registered model 'HeartDiseaseRandomForestTuned' already exists. Creating a new version of this model...
2026/04/01 06:56:34 WARNING mlflow.tracking._model_registry.fluent: Run with id 422307d7b79d47648248e36a99438a17 has no artifacts at artifact path 'randomforest_tuned_20260330T220816Z_a71ed44a', registering model based on models:/m-26c086216da24ed1bf7f56e300810075 instead
2026/04/01 06:56:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: HeartDiseaseRandomForestTuned, version 2
Created version '2' of model 'HeartDiseaseRandomForestTuned'.


Registration requested. Version: 2
Final status: READY
Alias not set (this may be expected on older MLflow versions): INVALID_PARAMETER_VALUE: 'latest' alias name (case insensitive) is reserved.
Registered model name: HeartDiseaseRandomForestTuned
Registered model version: 2
Source run id: 422307d7b79d47648248e36a99438a17
